# 向量数据库

## 连接数据库并创建集合

In [2]:
!pip install weaviate-client

  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.5/619.5 kB 318.6 kB/s  0:00:03eta 0:00:01
Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 294.2 kB/s  0:00:06 eta 0:00:01
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
  Attempting uninstall: typing-inspection
    Found existing installation: typing-inspection 0.4.0
    Uninstalling typing-inspection-0.4.0:
      Successfully uninstalled typing-inspection-0.4.0
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.33.2
    Uninstalling pydantic_core-2.33.2:
      Successfully uninstalled pydantic_core-2.33.2
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.11.7
    Uninstalling pydantic-2.11.7:
      Successfully uninstalled pydantic-2.11.7
   ━━━━━━━━━━━━━━━━━━━━━━

In [8]:
from openai import OpenAI

import helper
import importlib
importlib.reload(helper)
from helper import get_qwen_client

In [ ]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType
from weaviate.auth import AuthApiKey

client = weaviate.connect_to_local(
    host="localhost",
    port=8080,
    grpc_port=50051,
)

client.collections.create( # 创建集合
    "Article",
    vectorizer_config=Configure.Vectorizer.text2vec_openai(), # 指定向量化器 （指明了哪个嵌入模型，为添加的每篇文章生成语义向量）
    properties=[ # properties configuration is optional
        Property(name="title", data_type=DataType.TEXT),
        Property(name="body", data_type=DataType.TEXT), # 指定属性(集合里面存储新闻文章的标题和正文 及数据类型)
        ]
)

## 向集合添加对象

In [ ]:
with collection.batch.fixed_size(batch_size=200) as batch: # 批量添加数据
    for data_row in data_rows:
        batch.add.object(
            properties=data_row,
        )
        if batch.number_errors > 10:
            print("Batch import stopped due to excessive errors.") # 检测错误
            break

failed_objects = collection.batch.failed_objects # 处理失败
if failed_objects:
    print(f"Number of failed import:{len(failed_objects)}")
    print(f"First failed object:{failed_objects[0]}")

## 向量搜索实战

In [ ]:
from weaviate.classes.query import  MetadataQuery

articles = client.collections.get("Acticle") # 指定集合
response = articles.query.near_text( # 使用“近文本”执行向量搜索
    query="hotel capacity in downtown Vancouer",
    limit=2,
    return_metadata=MetadataQuery(distance=True) # 询问向量距离
)

for o in response.objects:
    print(o.properties)
    print(o.metadata.distance)

## 关键词搜索实战

In [ ]:
response = articles.query.bm25( # BM25关键词搜索
    query="Vancouver hotel capacity",
    limit=3
)

for o in response.objects:
    print(o.properties)

## 混合搜索实战

In [ ]:
response = articles.query.hybrid( # 混合搜索
    query="Vancouver hotel capacity",
    alpha=0.25, # 混合搜索权重为25%向量，75%关键词
    limit=3
)

for o in response.objects:
    print(o.properties)

## 筛选搜索进行中

In [ ]:
from warnings import filters
response = articles.query.hybrid( # 混合搜索
    query="History of urban development in Vancouver",
    filters = Filter.by_property('title').contains_any(['Vancouver']), # 添加元数据过滤器
    alpha=0.25, # 混合搜索权重为25%向量，75%关键词
    limit=4
)

for o in response.objects:
    print(o.properties)